# DeepLearning.AI

## OpenAI Function Calling in LangChain

<img src="./pics/functions-tools-agents-langchain.png" width=800> 

https://learn.deeplearning.ai/courses/functions-tools-agents-langchain/information

In [1]:
pwd

'c:\\Users\\crodr\\aiDeepLearning\\functions-tools-agents-langchain'

In [2]:
!python --version


Python 3.13.13


### OpenAI Function Calling in LangChain

* We are going to use the model `gpt-4o-mini`.

In [ ]:
import os
import openai
from dotenv import load_dotenv

load_dotenv()
openai.api_key = os.environ["OPENAI_API_KEY"]


In [5]:
from typing import List
from pydantic import BaseModel, Field

### Pydantic Syntax

Pydantic data classes are a blend of Python's data classes with the validation power of Pydantic.

They offer a concise way to define data structures whil ensuring that the data adheres to specified types and constraints.

In standard python you would create a class like this:

In [9]:
class User:
    def __init__(self, name: str, age: int, email: str):
        self.name = name
        self.age = age
        self.email = email

In [10]:
foo = User(name="Joe", age=32, email="joe@gmail.com")

In [11]:
foo.name

'Joe'

In [ ]:
foo = User(name="Joe", age="bar", email="joe@gmail.com")

In [13]:
foo.age

'bar'

In [ ]:
class pUser(BaseModel):
    name: str
    age: int
    email: str

In [15]:
foo_p = pUser(name="Jane", age=32, email="jane@gmail.com")

In [17]:
foo_p.name

'Jane'

**Note:** The next line is expected to fail.

In [18]:
foo_p = pUser(name="Jane", age="bar", email="jane@gmail.com")


ValidationError: 1 validation error for pUser
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='bar', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

In [19]:
class Class(BaseModel):
    students: List[pUser]

In [ ]:
obj = Class(students=[pUser(name="Jane", age=32, email="jane@gmail.com")])

In [21]:
obj

Class(students=[pUser(name='Jane', age=32, email='jane@gmail.com')])

### Pydantic to OpenAI function definition

In [ ]:
class WeatherSearch(BaseModel):
    """Call this with an airport code to get the weather at that airport"""

    airport_code: str = Field(description="airport code to get weather for")

In [25]:
# from langchain.utils.openai_functions import convert_pydantic_to_openai_function
from langchain_core.utils.function_calling import convert_to_openai_function

In [26]:
weather_function = convert_to_openai_function(WeatherSearch)

In [27]:
weather_function

{'name': 'WeatherSearch',
 'description': 'Call this with an airport code to get the weather at that airport',
 'parameters': {'properties': {'airport_code': {'description': 'airport code to get weather for',
    'type': 'string'}},
  'required': ['airport_code'],
  'type': 'object'}}

In [ ]:
class WeatherSearch1(BaseModel):
    airport_code: str = Field(description="airport code to get weather for")

**Note:** The next cell is expected to generate an error. As the comment string is missing, but this version don't generate the error, but the description is missing what will confuse the model.

In [31]:
convert_to_openai_function(WeatherSearch1)

{'name': 'WeatherSearch1',
 'description': '',
 'parameters': {'properties': {'airport_code': {'description': 'airport code to get weather for',
    'type': 'string'}},
  'required': ['airport_code'],
  'type': 'object'}}

In [ ]:
class WeatherSearch2(BaseModel):
    """Call this with an airport code to get the weather at that airport"""

    airport_code: str

In [33]:
convert_to_openai_function(WeatherSearch2)


{'name': 'WeatherSearch2',
 'description': 'Call this with an airport code to get the weather at that airport',
 'parameters': {'properties': {'airport_code': {'type': 'string'}},
  'required': ['airport_code'],
  'type': 'object'}}

In [35]:
# from langchain.chat_models import ChatOpenAI
from langchain_openai import ChatOpenAI


In [36]:
model = ChatOpenAI(model="gpt-4o-mini")


In [37]:
model.invoke("what is the weather in SF today?", functions=[weather_function])

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"airport_code":"SFO"}', 'name': 'WeatherSearch'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 68, 'total_tokens': 85, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f9748774de', 'id': 'chatcmpl-DdwCkgpGZKLAMXQ9kjt4Gargjhw5L', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='lc_run--019e118c-c9a8-7001-a6cb-6a23e5c0cdd8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 68, 'output_tokens': 17, 'total_tokens': 85, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [38]:
model_with_function = model.bind(functions=[weather_function])

In [39]:
model_with_function.invoke("what is the weather in sf?")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"airport_code":"SFO"}', 'name': 'WeatherSearch'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 67, 'total_tokens': 84, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_aa5a83636e', 'id': 'chatcmpl-DdwFF8CRVFtc4brwoHNGOeVmvQKtX', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='lc_run--019e118f-2d75-7891-b047-108d84ef0e38-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 67, 'output_tokens': 17, 'total_tokens': 84, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### Forcing it to use a function

We can force the model to use a function

In [ ]:
model_with_forced_function = model.bind(
    functions=[weather_function], function_call={"name": "WeatherSearch"}
)

In [41]:
model_with_forced_function.invoke("what is the weather in sf?")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"airport_code":"SFO"}', 'name': 'WeatherSearch'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 77, 'total_tokens': 84, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e19c763214', 'id': 'chatcmpl-DdwJmTHg0GnOkRZFAJdIw5ThkQmfn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e1193-7707-7063-a5a8-e4d82336701a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 77, 'output_tokens': 7, 'total_tokens': 84, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [42]:
model_with_forced_function.invoke("hi!")


AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"airport_code":"JFK"}', 'name': 'WeatherSearch'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 72, 'total_tokens': 79, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_fcf1a634a5', 'id': 'chatcmpl-DdwKODteJWfqy7sbStPDTSwfKwT9N', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e1194-0a16-7451-902b-3a873c597a34-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 7, 'total_tokens': 79, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### Using in a chain

We can use this model bound to function in a chain as we normally would

In [43]:
# from langchain.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatPromptTemplate


In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [("system", "You are a helpful assistant"), ("user", "{input}")]
)

In [45]:
chain = prompt | model_with_function

In [46]:
chain.invoke({"input": "what is the weather in sf?"})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"airport_code":"SFO"}', 'name': 'WeatherSearch'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 73, 'total_tokens': 90, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_134a436b39', 'id': 'chatcmpl-DdwPpK3Zr8AmsgSagry68mvWCVzZW', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='lc_run--019e1199-31db-7962-89d8-7788f4e83c02-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 73, 'output_tokens': 17, 'total_tokens': 90, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### Using multiple functions

Even better, we can pass a set of function and let the LLM decide which to use based on the question context.

In [ ]:
class ArtistSearch(BaseModel):
    """Call this to get the names of songs by a particular artist"""

    artist_name: str = Field(description="name of the artist to look up")
    n: int = Field(description="number of results")

In [48]:
functions = [
    convert_to_openai_function(WeatherSearch),
    convert_to_openai_function(ArtistSearch),
]

In [49]:
model_with_functions = model.bind(functions=functions)

In [50]:
model_with_functions.invoke("what is the weather in sf?")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"airport_code":"SFO"}', 'name': 'WeatherSearch'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 112, 'total_tokens': 129, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_d88f6e55bb', 'id': 'chatcmpl-DdwUsyChUyfY9aabSHcglhT1GEfCR', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='lc_run--019e119d-f391-7c91-bf8a-c7709e5cfcce-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 112, 'output_tokens': 17, 'total_tokens': 129, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [51]:
model_with_functions.invoke("what are three songs by taylor swift?")


AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"artist_name":"Taylor Swift","n":3}', 'name': 'ArtistSearch'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 114, 'total_tokens': 135, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c1582c9cfb', 'id': 'chatcmpl-DdwVNil7E6WPiAasAtc01VKgfM6z9', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='lc_run--019e119e-701c-74f0-bafb-e3484b1f1e7b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 114, 'output_tokens': 21, 'total_tokens': 135, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [52]:
model_with_functions.invoke("hi!")


AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 107, 'total_tokens': 117, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1b483c1ec4', 'id': 'chatcmpl-DdwVpT6yopXFbPEBtJdfhPTo18VAF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e119e-d916-7cf0-9239-9350751d36c9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 107, 'output_tokens': 10, 'total_tokens': 117, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})